# Laboratorio: Optimización de Circuitos con ZX-Calculus

Exploración práctica de la optimización de circuitos mediante PyZX y Qiskit.

**Objetivos:**
- Demostrar la equivalencia de circuitos usando el ZX-Calculus
- Medir la reducción de CNOTs con optimización ZX sobre circuitos VQE/QAOA
- Comparar la fidelidad del circuito original vs optimizado
- Analizar la relación entre T-count y recursos de computación tolerante a fallos

**Nota:** Este laboratorio muestra las técnicas con y sin PyZX (que puede no estar instalado).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, process_fidelity
from qiskit.circuit.library import EfficientSU2

# Verificar si PyZX está disponible
try:
    import pyzx as zx
    PYZX_AVAILABLE = True
    print(f'PyZX disponible: versión {zx.__version__}')
except ImportError:
    PYZX_AVAILABLE = False
    print('PyZX no instalado. Instalar con: pip install pyzx')
    print('Ejecutando análisis sin PyZX (demostraciones manuales)')

print('Qiskit importado correctamente')

## 1. Identidades del ZX-Calculus: verificación algebraica

In [ ]:
# Regla 1: Spider Fusion — Rz(α)·Rz(β) = Rz(α+β)
alpha, beta = np.pi/4, np.pi/3

qc1 = QuantumCircuit(1)
qc1.rz(alpha, 0)
qc1.rz(beta, 0)

qc2 = QuantumCircuit(1)
qc2.rz(alpha + beta, 0)

op1 = Operator(qc1)
op2 = Operator(qc2)
print(f'Spider Fusion: Rz({alpha:.3f})·Rz({beta:.3f}) = Rz({alpha+beta:.3f})?')
print(f'  Matrices iguales: {np.allclose(op1.data, op2.data)}')

# Regla 2: H·Z·H = X
qc_hzh = QuantumCircuit(1)
qc_hzh.h(0); qc_hzh.z(0); qc_hzh.h(0)

qc_x = QuantumCircuit(1)
qc_x.x(0)

print(f'\nH·Z·H = X?')
print(f'  Matrices iguales: {np.allclose(Operator(qc_hzh).data, Operator(qc_x).data)}')

# Regla 3: CNOT = (I⊗H)·CZ·(I⊗H)
qc_cnot = QuantumCircuit(2)
qc_cnot.cx(0, 1)

qc_cz_h = QuantumCircuit(2)
qc_cz_h.h(1); qc_cz_h.cz(0, 1); qc_cz_h.h(1)

print(f'\nCNOT = (I⊗H)·CZ·(I⊗H)?')
print(f'  Matrices iguales: {np.allclose(Operator(qc_cnot).data, Operator(qc_cz_h).data)}')

# Regla 4: ZZ se cancela
qc_zz = QuantumCircuit(1)
qc_zz.z(0); qc_zz.z(0)

qc_id = QuantumCircuit(1)  # identidad

print(f'\nZ·Z = I?')
print(f'  Matrices iguales: {np.allclose(Operator(qc_zz).data, Operator(qc_id).data)}')

## 2. Optimización Manual de un Circuito Redundante

In [ ]:
# Circuito con redundancias que el ZX-Calculus puede simplificar
qc_redundante = QuantumCircuit(2)
qc_redundante.h(0)
qc_redundante.cx(0, 1)
qc_redundante.cx(0, 1)  # CNOT·CNOT = I → se cancela
qc_redundante.h(0)
qc_redundante.h(0)      # H·H = I → se cancela
qc_redundante.cx(0, 1)

# Circuito optimizado equivalente
qc_optimizado = QuantumCircuit(2)
qc_optimizado.h(0)
qc_optimizado.cx(0, 1)

# Verificar equivalencia
op_red = Operator(qc_redundante)
op_opt = Operator(qc_optimizado)
son_iguales = np.allclose(op_red.data, op_opt.data)

print('Circuito redundante:')
print(qc_redundante.draw())
print(f'\nPuertas: {dict(qc_redundante.count_ops())}')
print(f'Profundidad: {qc_redundante.depth()}')

print('\nCircuito optimizado:')
print(qc_optimizado.draw())
print(f'Puertas: {dict(qc_optimizado.count_ops())}')
print(f'Profundidad: {qc_optimizado.depth()}')

print(f'\n¿Unitariamente equivalentes? {son_iguales}')
print(f'Reducción de CNOTs: {qc_redundante.count_ops()["cx"]} → {qc_optimizado.count_ops()["cx"]} ({100*(1 - qc_optimizado.count_ops()["cx"]/qc_redundante.count_ops()["cx"]):.0f}% menos)')

## 3. Optimización con PyZX (si está disponible)

In [ ]:
if PYZX_AVAILABLE:
    from qiskit.qasm2 import dumps
    
    # Circuito VQE parametrizado con valores fijos
    n_qubits = 4
    ansatz = EfficientSU2(n_qubits, reps=2, entanglement='linear')
    params = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)
    bound = ansatz.assign_parameters(params)
    
    # Estadísticas originales
    cx_original = bound.count_ops().get('cx', 0)
    depth_original = bound.depth()
    
    print(f'Circuito VQE EfficientSU2 ({n_qubits} qubits, reps=2):')
    print(f'  CNOTs originales: {cx_original}')
    print(f'  Profundidad original: {depth_original}')
    
    # Convertir a PyZX y optimizar
    qasm_str = dumps(bound)
    pyzx_circ = zx.Circuit.from_qasm(qasm_str)
    g = pyzx_circ.to_graph()
    
    print(f'\nGrafo ZX:')
    print(f'  Vértices: {g.num_vertices()}')
    print(f'  Aristas: {g.num_edges()}')
    
    zx.full_reduce(g)
    print(f'\nGrafo ZX simplificado:')
    print(f'  Vértices: {g.num_vertices()}')
    print(f'  Aristas: {g.num_edges()}')
    
    optimized = zx.extract_circuit(g)
    cx_optimized = optimized.count_2q_gates()
    
    print(f'\nResultados:')
    print(f'  CNOTs originales: {cx_original}')
    print(f'  CNOTs optimizados: {cx_optimized}')
    print(f'  Reducción: {100*(1 - cx_optimized/cx_original):.1f}%')

else:
    # Demostración con Qiskit transpiler
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    
    n_qubits = 4
    ansatz = EfficientSU2(n_qubits, reps=2, entanglement='linear')
    params = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)
    bound = ansatz.assign_parameters(params)
    
    cx_original = bound.count_ops().get('cx', 0)
    
    # Optimización con el transpilador de Qiskit (nivel 3)
    pm = generate_preset_pass_manager(optimization_level=3, basis_gates=['cx', 'u'])
    optimized_qc = pm.run(bound)
    cx_optimized = optimized_qc.count_ops().get('cx', 0)
    
    print(f'Optimización con Qiskit Transpiler (nivel 3):')
    print(f'  CNOTs originales: {cx_original}')
    print(f'  CNOTs tras optimización: {cx_optimized}')
    reduction = 100*(1 - cx_optimized/cx_original) if cx_original > 0 else 0
    print(f'  Reducción: {reduction:.1f}%')
    print(f'  (Para reducción ZX completa, instalar PyZX: pip install pyzx)')

## 4. T-count y Recursos de Corrección de Errores

In [ ]:
# El T-count (número de puertas T = Rz(π/4)) determina el coste en
# destilación de estados mágicos para la computación tolerante a fallos

def count_t_gates(circuit: QuantumCircuit) -> int:
    """Cuenta el número de puertas T (y T†) en un circuito."""
    ops = circuit.count_ops()
    return ops.get('t', 0) + ops.get('tdg', 0)

# Implementaciones de la puerta Toffoli con diferentes T-counts
# Estándar: 7 puertas T
tof_standard = QuantumCircuit(3)
tof_standard.h(2)
tof_standard.cx(1, 2)
tof_standard.tdg(2)
tof_standard.cx(0, 2)
tof_standard.t(2)
tof_standard.cx(1, 2)
tof_standard.tdg(2)
tof_standard.cx(0, 2)
tof_standard.t(1)
tof_standard.t(2)
tof_standard.h(2)
tof_standard.cx(0, 1)
tof_standard.t(0)
tof_standard.tdg(1)
tof_standard.cx(0, 1)

t_count_standard = count_t_gates(tof_standard)
print(f'Toffoli estándar: T-count = {t_count_standard}')

# Relación T-count con recursos de corrección de errores
print('\nCoste de T-count en corrección de errores:')
print(f'  Cada puerta T requiere ~distillation de estados mágicos')
print(f'  Con p_error = 1e-3 y T-count = {t_count_standard}:')
t_counts = [1, 7, 10, 100, 1000]
qubits_per_T = 15  # overhead de destilación de primer nivel
print(f'\n  {'T-count':<12} {'Qubits ancilla (estim.)'}' )
print('  ' + '-' * 35)
for T in t_counts:
    qubits_ancilla = T * qubits_per_T
    print(f'  {T:<12} {qubits_ancilla}')

print(f'\nEl ZX-Calculus puede reducir el T-count, reduciendo directamente')
print(f'el overhead de destilación de estados mágicos.')